# Restaurant Order Agent — Sprint 1 demo

This notebook shows the agent in two modes:

- **Groq mode:** Qwen on Groq. Needs a free API key from https://console.groq.com.
- **Local mode:** Qwen3.5-4B in 4-bit on the Colab GPU.

**Before you start in Colab:**

1. Add your key as a Colab secret: click the key icon on the left, add `GROQ_API_KEY`, and turn on notebook access.
2. For local mode only: select **Runtime → Change runtime type → T4 GPU**.

The service day is fixed to Friday 2026-11-06, as in the evaluation.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/ai-agent-project"):
        !git clone -q https://github.com/RZ-Metropolia/ai-agent-project.git /content/ai-agent-project
    %cd /content/ai-agent-project
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    %cd ..

In [ ]:
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("GROQ_API_KEY"):
    if IN_COLAB:
        from google.colab import userdata
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    else:
        from getpass import getpass
        os.environ["GROQ_API_KEY"] = getpass("Groq API key: ")

## Setup

A fresh database from the seed file, and the mock inventory API in the background.

In [ ]:
from datetime import date
from zoneinfo import ZoneInfo

from src.agent import build_model, print_trace, run_agent
from src.agent_tools import make_tools
from src.config import ToolContext
from src.db import init_db
from src.inventory_api import BackgroundServer
from src.inventory_client import InventoryClient, set_inventory_mode

db_path = init_db()
server = BackgroundServer().start()
ctx = ToolContext(
    db_path=db_path,
    inventory=InventoryClient(server.url),
    service_date=date(2026, 11, 6),
    timezone=ZoneInfo("Europe/Berlin"),
)
tools = make_tools(ctx)


def ask(model, request):
    result = run_agent(model, ctx, tools, request)
    print("Request:", request)
    print("Tool calls:")
    print_trace(result)
    print("\nAnswer:", result.answer)
    return result

## Demo A — a vegetarian order with Groq

The trace shows each tool call, its arguments, and the start of its result.

In [ ]:
groq = build_model("groq")
ask(groq, "Prepare a vegetarian pickup order for one person at 18:30: "
          "one main course and one drink, with a total of at most 20 euros.");

## Demo B — the inventory API is down

The outage switch makes the API return HTTP 503. The agent must not save a draft.

In [ ]:
set_inventory_mode(server.url, "outage")
ask(groq, "Please draft a pickup order for one Margherita Pizza at 18:30.")
set_inventory_mode(server.url, "normal")

## Evaluation with Groq (3 runs per scenario)

Each scenario gets a fresh database. The details (answers, checks, tool traces) are saved in `docs/eval_results/`.

In [ ]:
from src.evaluation import run_evaluation

groq_results = run_evaluation(groq, "groq", runs=3)

## Local mode — Qwen3.5-4B in 4-bit

Needs the GPU runtime. Loading the model takes a few minutes.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
local = build_model("local")
ask(local, "Prepare a vegetarian pickup order for one person at 18:30: "
           "one main course and one drink, with a total of at most 20 euros.");

In [ ]:
local_results = run_evaluation(local, "local", runs=1)

## Save the results

Download the result files and add them to `docs/eval_results/` in the repository. Then fill in `docs/SPRINT_1_EVALUATION.md`. Read each answer before you copy a result: the automatic checks only look for key words.

In [ ]:
if IN_COLAB:
    from google.colab import files
    !zip -q -r eval_results.zip docs/eval_results
    files.download("eval_results.zip")